# Experiment 14: Drift Detection Under Gradual and Recurring Distribution Shifts

## 1. Objective
Evaluate how embedding-level drift detectors respond to abrupt, gradual, and recurring distribution shifts using controlled synthetic embedding streams.

## 2. Research Questions
1. How does the temporal pattern of drift affect detection rate and detection delay?
2. How frequently do detectors raise false alarms during stable periods?
3. How consistently do Euclidean and regularized Mahalanobis detectors perform across different drift patterns?
4. Does recurring drift produce different detection behaviour from a single abrupt shift?

## 3. Experimental Design
The experiment will compare three controlled distribution-shift conditions:

- **Abrupt drift:** The embedding distribution changes suddenly at a predefined time.
- **Gradual drift:** The embedding distribution transitions progressively from the reference distribution to a shifted distribution.
- **Recurring drift:** The embedding distribution alternates between the reference distribution and a previously observed shifted distribution.

The detector implementations and applicable frozen configurations from previous experiments will be reused where appropriate. Any necessary changes will be documented before evaluation.

## 4. Evaluation Metrics
- Detection rate
- Detection delay, measured in evaluation windows
- False-alarm rate during stable periods
- Missed-detection rate
- Variability across independent random seeds

## 5. Reproducibility
All random seeds, drift parameters, detector settings, thresholds, and evaluation procedures will be documented. Experiment 14 outputs will be stored separately from previous experiment results.

## 6. Scientific Limitations
This experiment evaluates controlled synthetic distribution shifts. Its results will not, by themselves, establish real-world biometric recognition accuracy, fairness, or deployment readiness. Detection-onset definitions and event-matching rules will be specified before calculating evaluation metrics.

## 7. Relationship to Previous Experiments
Experiment 14 is intended to extend the existing embedding-level drift-detection evaluation by investigating temporal drift patterns. Previously completed multi-seed variance analyses will not be repeated without a distinct research justification.

In [1]:

# Experiment 14: Environment Setup

import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
BASE_SEED = 202614
np.random.seed(BASE_SEED)

# Separate output directory: do not overwrite previous experiments
OUTPUT_DIR = Path("results/experiment_14")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Record environment information
environment = {
    "experiment": 14,
    "title": "Drift Detection Under Gradual and Recurring Distribution Shifts",
    "random_seed": BASE_SEED,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

print("Experiment 14 environment initialized.")
print(json.dumps(environment, indent=2))
print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")

# Save environment metadata
with open(OUTPUT_DIR / "environment.json", "w") as f:
    json.dump(environment, f, indent=2)

print("Environment metadata saved.")


Experiment 14 environment initialized.
{
  "experiment": 14,
  "title": "Drift Detection Under Gradual and Recurring Distribution Shifts",
  "random_seed": 202614,
  "python_version": "3.13.16",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "numpy_version": "2.1.3",
  "pandas_version": "2.2.3"
}

Output directory: /content/results/experiment_14
Environment metadata saved.


In [2]:

# Experiment 14: Define Proposed Drift Conditions

DRIFT_CONDITIONS = {
    "abrupt": {
        "description": "A sudden shift from the reference distribution",
        "transition_windows": 1
    },
    "gradual": {
        "description": "A progressive transition to a shifted distribution",
        "transition_windows": 10
    },
    "recurring": {
        "description": "Alternation between reference and shifted distributions",
        "transition_windows": 1
    }
}

print("Proposed drift conditions:")
for name, settings in DRIFT_CONDITIONS.items():
    print(f"\n{name.upper()}")
    print("Description:", settings["description"])
    print("Transition windows:", settings["transition_windows"])

print("\nDesign check complete. No data generated.")


Proposed drift conditions:

ABRUPT
Description: A sudden shift from the reference distribution
Transition windows: 1

GRADUAL
Description: A progressive transition to a shifted distribution
Transition windows: 10

RECURRING
Description: Alternation between reference and shifted distributions
Transition windows: 1

Design check complete. No data generated.


In [3]:

# Experiment 14: Define Temporal Evaluation Schedule

N_WINDOWS = 60
WINDOW_SIZE = 20

# Zero-based window indices
DRIFT_SCHEDULE = {
    "abrupt": [
        {"start": 20, "end": 60, "type": "shifted"}
    ],
    "gradual": [
        {"start": 20, "end": 30, "type": "transition"},
        {"start": 30, "end": 60, "type": "shifted"}
    ],
    "recurring": [
        {"start": 20, "end": 30, "type": "shifted"},
        {"start": 30, "end": 40, "type": "baseline"},
        {"start": 40, "end": 50, "type": "shifted"},
        {"start": 50, "end": 60, "type": "baseline"}
    ]
}

print("Total windows:", N_WINDOWS)
print("Samples per window:", WINDOW_SIZE)
print("Total samples per stream:", N_WINDOWS * WINDOW_SIZE)

for condition, segments in DRIFT_SCHEDULE.items():
    print(f"\n{condition.upper()}")
    for segment in segments:
        print(segment)

print("\nSchedule defined. No synthetic data generated.")


Total windows: 60
Samples per window: 20
Total samples per stream: 1200

ABRUPT
{'start': 20, 'end': 60, 'type': 'shifted'}

GRADUAL
{'start': 20, 'end': 30, 'type': 'transition'}
{'start': 30, 'end': 60, 'type': 'shifted'}

RECURRING
{'start': 20, 'end': 30, 'type': 'shifted'}
{'start': 30, 'end': 40, 'type': 'baseline'}
{'start': 40, 'end': 50, 'type': 'shifted'}
{'start': 50, 'end': 60, 'type': 'baseline'}

Schedule defined. No synthetic data generated.


In [4]:

# Experiment 14: Synthetic Distribution Configuration
# Configuration only — no data generated in this step.

SYNTHETIC_CONFIG = {
    "random_seed": 202614,
    "n_windows": N_WINDOWS,
    "window_size": WINDOW_SIZE,
    "embedding_dimension": 64,
    "reference_mean": 0.0,
    "reference_std": 1.0,
    "mean_shift_strength": 0.5,
    "gradual_transition_windows": 10,
    "drift_start_window": 20,
    "recurring_shifted_windows": [20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                                  40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
    "detectors": ["euclidean", "mahalanobis"]
}

print("Synthetic configuration:")
for key, value in SYNTHETIC_CONFIG.items():
    print(f"{key}: {value}")

print("\nConfiguration defined. No synthetic data generated.")


Synthetic configuration:
random_seed: 202614
n_windows: 60
window_size: 20
embedding_dimension: 64
reference_mean: 0.0
reference_std: 1.0
mean_shift_strength: 0.5
gradual_transition_windows: 10
drift_start_window: 20
recurring_shifted_windows: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
detectors: ['euclidean', 'mahalanobis']

Configuration defined. No synthetic data generated.


In [5]:

# Experiment 14: Configuration Validation
# Validation only — no data generation or detector evaluation.

assert SYNTHETIC_CONFIG["n_windows"] == N_WINDOWS
assert SYNTHETIC_CONFIG["window_size"] == WINDOW_SIZE
assert SYNTHETIC_CONFIG["embedding_dimension"] > 0
assert SYNTHETIC_CONFIG["mean_shift_strength"] > 0
assert SYNTHETIC_CONFIG["gradual_transition_windows"] == 10
assert SYNTHETIC_CONFIG["drift_start_window"] == 20

assert SYNTHETIC_CONFIG["detectors"] == [
    "euclidean", "mahalanobis"
]

# Verify recurring drift windows fall within the stream.
recurring_windows = SYNTHETIC_CONFIG["recurring_shifted_windows"]

assert all(0 <= w < N_WINDOWS for w in recurring_windows)
assert len(recurring_windows) == len(set(recurring_windows))

# Verify the recurring schedule matches the configuration.
expected_recurring = set(range(20, 30)) | set(range(40, 50))
assert set(recurring_windows) == expected_recurring

print("All configuration checks passed.")
print("Windows:", N_WINDOWS)
print("Window size:", WINDOW_SIZE)
print("Embedding dimension:", SYNTHETIC_CONFIG["embedding_dimension"])
print("Detectors:", ", ".join(SYNTHETIC_CONFIG["detectors"]))
print("Recurring drift windows:", len(recurring_windows))
print("No synthetic data generated.")


All configuration checks passed.
Windows: 60
Window size: 20
Embedding dimension: 64
Detectors: euclidean, mahalanobis
Recurring drift windows: 20
No synthetic data generated.


## Step 7: Data-Generation and Calibration Protocol

### Reference Distribution
Synthetic reference embeddings will be sampled from a multivariate normal distribution. The reference distribution will be identical across all three drift conditions.

### Drift Conditions
- **Abrupt:** Apply the full distribution shift at the predefined onset.
- **Gradual:** Increase the shift progressively over 10 consecutive windows.
- **Recurring:** Alternate between the reference distribution and the shifted distribution according to the predefined schedule.

### Calibration and Evaluation
Detector thresholds will be calibrated using data separate from the evaluation streams. Thresholds will be frozen before the final evaluation. Calibration and evaluation random seeds will be recorded.

### Evaluation
The Euclidean and regularized Mahalanobis detectors will be compared using detection rate, detection delay, false-alarm rate, and missed-detection rate. Results will be summarized across independent random seeds.

### Reproducibility
All generation parameters, detector settings, calibration thresholds, and raw results will be saved separately under `results/experiment_14/`.

In [6]:

# Experiment 14: Synthetic Stream Generator

import numpy as np
import pandas as pd

def generate_drift_stream(condition, seed, config):
    """
    Generate a synthetic embedding stream with a specified
    temporal drift pattern.

    Returns:
        X: (n_windows, window_size, embedding_dimension)
        labels: true distribution state for each window
    """
    rng = np.random.default_rng(seed)

    n_windows = config["n_windows"]
    window_size = config["window_size"]
    dimension = config["embedding_dimension"]
    shift_strength = config["mean_shift_strength"]
    transition_windows = config["gradual_transition_windows"]

    # Reference distribution: independent standard-normal dimensions
    X = rng.normal(
        loc=config["reference_mean"],
        scale=config["reference_std"],
        size=(n_windows, window_size, dimension)
    )

    # Fixed shift direction for comparable conditions
    direction_rng = np.random.default_rng(202614)
    direction = direction_rng.normal(size=dimension)
    direction = direction / np.linalg.norm(direction)
    shift = shift_strength * np.sqrt(dimension) * direction

    labels = np.zeros(n_windows, dtype=float)

    if condition == "abrupt":
        X[20:] += shift
        labels[20:] = 1.0

    elif condition == "gradual":
        start = config["drift_start_window"]
        end = start + transition_windows

        for w in range(start, end):
            fraction = (w - start + 1) / transition_windows
            X[w] += fraction * shift
            labels[w] = fraction

        X[end:] += shift
        labels[end:] = 1.0

    elif condition == "recurring":
        for w in config["recurring_shifted_windows"]:
            X[w] += shift
            labels[w] = 1.0

    else:
        raise ValueError(f"Unknown condition: {condition}")

    return X, labels


print("Synthetic stream generator defined.")
print("No streams generated yet.")


Synthetic stream generator defined.
No streams generated yet.


In [7]:

# Experiment 14: Generator Validation

conditions = ["abrupt", "gradual", "recurring"]
validation_results = []

for condition in conditions:
    X_test, labels_test = generate_drift_stream(
        condition=condition,
        seed=202614,
        config=SYNTHETIC_CONFIG
    )

    assert X_test.shape == (
        N_WINDOWS,
        WINDOW_SIZE,
        SYNTHETIC_CONFIG["embedding_dimension"]
    )
    assert labels_test.shape == (N_WINDOWS,)
    assert np.isfinite(X_test).all()
    assert np.isfinite(labels_test).all()

    validation_results.append({
        "condition": condition,
        "shape": X_test.shape,
        "finite_values": bool(np.isfinite(X_test).all()),
        "shifted_windows": int(np.sum(labels_test > 0)),
        "max_shift_intensity": float(labels_test.max())
    })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

print("Generator validation passed.")
print("These are diagnostic streams, not final evaluation results.")


,condition,shape,finite_values,shifted_windows,max_shift_intensity
0,abrupt,"(60, 20, 64)",True,40,1.0
1,gradual,"(60, 20, 64)",True,40,1.0
2,recurring,"(60, 20, 64)",True,20,1.0


Generator validation passed.
These are diagnostic streams, not final evaluation results.


## Step 10: Detector Calibration Protocol

### Independent Calibration
Generate independent reference and calibration samples from the unchanged baseline distribution. Do not use evaluation streams to estimate detector thresholds.

### False-Alarm Control
Calibrate Euclidean and regularized Mahalanobis detectors against a predefined target false-alarm rate. Record the calibration sample size, random seeds, covariance regularization method, and resulting thresholds.

### Frozen Thresholds
Freeze all detector thresholds before evaluating abrupt, gradual, and recurring drift. Do not tune thresholds separately for each condition.

### Evaluation Separation
Use independent random seeds for calibration and evaluation. Report false alarms during baseline periods separately from detections during known shifted periods.

### Statistical Reporting
Aggregate results across independent runs and report uncertainty estimates. Distinguish detection of a new drift episode from alarms that remain active after an earlier episode.

In [8]:

# Experiment 14: Independent Baseline Calibration Data

CALIBRATION_CONFIG = {
    "reference_samples": 1000,
    "calibration_windows": 200,
    "window_size": WINDOW_SIZE,
    "embedding_dimension": SYNTHETIC_CONFIG["embedding_dimension"],
    "target_false_alarm_rate": 0.05,
    "reference_seed": 20261401,
    "calibration_seed": 20261402
}

dimension = CALIBRATION_CONFIG["embedding_dimension"]
window_size = CALIBRATION_CONFIG["window_size"]

reference_rng = np.random.default_rng(
    CALIBRATION_CONFIG["reference_seed"]
)
calibration_rng = np.random.default_rng(
    CALIBRATION_CONFIG["calibration_seed"]
)

# Independent baseline reference samples
X_reference = reference_rng.normal(
    loc=SYNTHETIC_CONFIG["reference_mean"],
    scale=SYNTHETIC_CONFIG["reference_std"],
    size=(CALIBRATION_CONFIG["reference_samples"], dimension)
)

# Independent calibration windows, all from the baseline distribution
X_calibration = calibration_rng.normal(
    loc=SYNTHETIC_CONFIG["reference_mean"],
    scale=SYNTHETIC_CONFIG["reference_std"],
    size=(
        CALIBRATION_CONFIG["calibration_windows"],
        window_size,
        dimension
    )
)

assert X_reference.shape == (
    CALIBRATION_CONFIG["reference_samples"], dimension
)
assert X_calibration.shape == (
    CALIBRATION_CONFIG["calibration_windows"],
    window_size,
    dimension
)
assert np.isfinite(X_reference).all()
assert np.isfinite(X_calibration).all()

print("Independent baseline data generated successfully.")
print("Reference shape:", X_reference.shape)
print("Calibration shape:", X_calibration.shape)
print("Target false-alarm rate:", CALIBRATION_CONFIG["target_false_alarm_rate"])
print("Reference seed:", CALIBRATION_CONFIG["reference_seed"])
print("Calibration seed:", CALIBRATION_CONFIG["calibration_seed"])
print("All values finite:", np.isfinite(X_reference).all() and np.isfinite(X_calibration).all())
print("No drift detector evaluated in this step.")


Independent baseline data generated successfully.
Reference shape: (1000, 64)
Calibration shape: (200, 20, 64)
Target false-alarm rate: 0.05
Reference seed: 20261401
Calibration seed: 20261402
All values finite: True
No drift detector evaluated in this step.


In [9]:

# Experiment 14: Detector Scoring Functions

from sklearn.covariance import LedoitWolf

# Reference statistics
reference_mean = X_reference.mean(axis=0)

# Fit a regularized covariance model on reference data
covariance_model = LedoitWolf().fit(X_reference)
precision_matrix = covariance_model.precision_

def score_euclidean(window, reference_center):
    """Euclidean distance between window mean and reference mean."""
    window_mean = window.mean(axis=0)
    return float(np.linalg.norm(window_mean - reference_center))

def score_mahalanobis(window, reference_center, precision):
    """Regularized Mahalanobis distance between window and reference means."""
    window_mean = window.mean(axis=0)
    difference = window_mean - reference_center
    squared_distance = difference @ precision @ difference
    return float(np.sqrt(max(squared_distance, 0.0)))

# Numerical-stability checks
assert reference_mean.shape == (dimension,)
assert precision_matrix.shape == (dimension, dimension)
assert np.isfinite(reference_mean).all()
assert np.isfinite(precision_matrix).all()
assert np.allclose(precision_matrix, precision_matrix.T, atol=1e-8)

test_window = X_calibration[0]
test_euclidean = score_euclidean(test_window, reference_mean)
test_mahalanobis = score_mahalanobis(
    test_window, reference_mean, precision_matrix
)

assert np.isfinite(test_euclidean)
assert np.isfinite(test_mahalanobis)

print("Detector scoring functions defined.")
print("Reference center shape:", reference_mean.shape)
print("Precision matrix shape:", precision_matrix.shape)
print("Covariance estimator: Ledoit-Wolf")
print("Diagnostic Euclidean score:", test_euclidean)
print("Diagnostic Mahalanobis score:", test_mahalanobis)
print("Numerical-stability checks passed.")
print("Thresholds not calibrated yet.")


Detector scoring functions defined.
Reference center shape: (64,)
Precision matrix shape: (64, 64)
Covariance estimator: Ledoit-Wolf
Diagnostic Euclidean score: 1.4484392704789661
Diagnostic Mahalanobis score: 1.4514818041533086
Numerical-stability checks passed.
Thresholds not calibrated yet.


In [10]:

# Experiment 14: Threshold Calibration and Freezing

# Score every independent baseline calibration window.
calibration_scores = []

for window_index, window in enumerate(X_calibration):
    calibration_scores.append({
        "window": window_index,
        "euclidean": score_euclidean(window, reference_mean),
        "mahalanobis": score_mahalanobis(
            window, reference_mean, precision_matrix
        )
    })

calibration_df = pd.DataFrame(calibration_scores)

target_fpr = CALIBRATION_CONFIG["target_false_alarm_rate"]

# Empirical upper-tail quantile.
# With finite calibration data, the realized FPR may differ from the target.
thresholds = {
    "euclidean": float(
        calibration_df["euclidean"].quantile(1 - target_fpr)
    ),
    "mahalanobis": float(
        calibration_df["mahalanobis"].quantile(1 - target_fpr)
    )
}

# Freeze the thresholds before evaluation.
FROZEN_THRESHOLDS = thresholds.copy()

print("Calibration complete.")
print("Calibration windows:", len(calibration_df))
print("Target false-alarm rate:", target_fpr)
print("\nFrozen thresholds:")

for detector, threshold in FROZEN_THRESHOLDS.items():
    print(f"{detector}: {threshold:.6f}")

print("\nThresholds are now frozen for the planned evaluation.")
print("Do not tune them using evaluation-stream results.")

# Save calibration scores and thresholds.
calibration_df.to_csv(
    OUTPUT_DIR / "calibration_scores.csv", index=False
)

with open(OUTPUT_DIR / "frozen_thresholds.json", "w") as f:
    json.dump({
        "target_false_alarm_rate": target_fpr,
        "calibration_windows": len(calibration_df),
        "thresholds": FROZEN_THRESHOLDS,
        "method": "Empirical upper-tail quantile",
        "reference_seed": CALIBRATION_CONFIG["reference_seed"],
        "calibration_seed": CALIBRATION_CONFIG["calibration_seed"]
    }, f, indent=2)

print("\nSaved calibration_scores.csv")
print("Saved frozen_thresholds.json")


Calibration complete.
Calibration windows: 200
Target false-alarm rate: 0.05

Frozen thresholds:
euclidean: 2.070293
mahalanobis: 2.074642

Thresholds are now frozen for the planned evaluation.
Do not tune them using evaluation-stream results.

Saved calibration_scores.csv
Saved frozen_thresholds.json


In [11]:

# Experiment 14: Independent Baseline Control Evaluation
# Thresholds remain frozen; no drift is introduced.

N_CONTROL_SEEDS = 20
CONTROL_WINDOWS = 500

control_results = []

for seed_offset in range(N_CONTROL_SEEDS):
    seed = 20261500 + seed_offset
    rng = np.random.default_rng(seed)

    control_stream = rng.normal(
        loc=SYNTHETIC_CONFIG["reference_mean"],
        scale=SYNTHETIC_CONFIG["reference_std"],
        size=(
            CONTROL_WINDOWS,
            WINDOW_SIZE,
            SYNTHETIC_CONFIG["embedding_dimension"]
        )
    )

    for detector in ["euclidean", "mahalanobis"]:
        scores = []

        for window in control_stream:
            if detector == "euclidean":
                score = score_euclidean(window, reference_mean)
            else:
                score = score_mahalanobis(
                    window, reference_mean, precision_matrix
                )
            scores.append(score)

        alarms = np.asarray(scores) > FROZEN_THRESHOLDS[detector]

        control_results.append({
            "seed": seed,
            "detector": detector,
            "n_windows": CONTROL_WINDOWS,
            "n_alarms": int(alarms.sum()),
            "false_alarm_rate": float(alarms.mean())
        })

control_df = pd.DataFrame(control_results)

control_summary = (
    control_df.groupby("detector")
    .agg(
        mean_false_alarm_rate=("false_alarm_rate", "mean"),
        sd_across_seeds=("false_alarm_rate", "std"),
        total_alarms=("n_alarms", "sum"),
        total_windows=("n_windows", "sum")
    )
    .reset_index()
)

display(control_summary)

control_df.to_csv(
    OUTPUT_DIR / "independent_control_results.csv", index=False
)
control_summary.to_csv(
    OUTPUT_DIR / "independent_control_summary.csv", index=False
)

print("Independent baseline evaluation complete.")
print("Thresholds unchanged:", FROZEN_THRESHOLDS)
print("Saved control results and summary.")


,detector,mean_false_alarm_rate,sd_across_seeds,total_alarms,total_windows
0,euclidean,0.0497,0.007794,497,10000
1,mahalanobis,0.0497,0.007794,497,10000


Independent baseline evaluation complete.
Thresholds unchanged: {'euclidean': 2.0702929416972333, 'mahalanobis': 2.0746417163536846}
Saved control results and summary.


In [12]:

# Experiment 14: Compare Baseline Detector Scores
# Diagnostic analysis only; frozen thresholds remain unchanged.

score_comparison = []

rng = np.random.default_rng(20261600)

diagnostic_stream = rng.normal(
    loc=SYNTHETIC_CONFIG["reference_mean"],
    scale=SYNTHETIC_CONFIG["reference_std"],
    size=(
        500,
        WINDOW_SIZE,
        SYNTHETIC_CONFIG["embedding_dimension"]
    )
)

for i, window in enumerate(diagnostic_stream):
    e_score = score_euclidean(window, reference_mean)
    m_score = score_mahalanobis(
        window, reference_mean, precision_matrix
    )

    score_comparison.append({
        "window": i,
        "euclidean_score": e_score,
        "mahalanobis_score": m_score,
        "euclidean_alarm": e_score > FROZEN_THRESHOLDS["euclidean"],
        "mahalanobis_alarm": m_score > FROZEN_THRESHOLDS["mahalanobis"]
    })

score_comparison_df = pd.DataFrame(score_comparison)

print("Correlation between detector scores:",
      score_comparison_df["euclidean_score"].corr(
          score_comparison_df["mahalanobis_score"]
      ))

print("Identical score count:",
      np.isclose(
          score_comparison_df["euclidean_score"],
          score_comparison_df["mahalanobis_score"]
      ).sum(), "out of", len(score_comparison_df))

print("Identical alarm decisions:",
      (
          score_comparison_df["euclidean_alarm"]
          == score_comparison_df["mahalanobis_alarm"]
      ).mean())

display(score_comparison_df.head())

score_comparison_df.to_csv(
    OUTPUT_DIR / "baseline_detector_score_comparison.csv",
    index=False
)

print("Saved baseline_detector_score_comparison.csv")


Correlation between detector scores: 1.0
Identical score count: 0 out of 500
Identical alarm decisions: 1.0


,window,euclidean_score,mahalanobis_score,euclidean_alarm,mahalanobis_alarm
0,0,1.903364,1.907362,False,False
1,1,1.934558,1.938621,False,False
2,2,1.883922,1.887879,False,False
3,3,1.852189,1.856080,False,False
4,4,1.986427,1.990599,False,False


Saved baseline_detector_score_comparison.csv


In [13]:

# Experiment 14: Evaluate Temporal Drift Conditions
# Uses frozen thresholds; no threshold tuning.

EVALUATION_SEEDS = range(20261700, 20261720)
evaluation_rows = []
trace_rows = []

for condition in ["abrupt", "gradual", "recurring"]:
    for seed in EVALUATION_SEEDS:
        X_stream, true_shift = generate_drift_stream(
            condition=condition,
            seed=seed,
            config=SYNTHETIC_CONFIG
        )

        for detector in ["euclidean", "mahalanobis"]:
            scores = []

            for window_index, window in enumerate(X_stream):
                if detector == "euclidean":
                    score = score_euclidean(window, reference_mean)
                else:
                    score = score_mahalanobis(
                        window, reference_mean, precision_matrix
                    )

                scores.append(score)

            scores = np.asarray(scores)
            alarms = scores > FROZEN_THRESHOLDS[detector]

            # A window is considered shifted if any shift is present.
            shifted_mask = true_shift > 0
            stable_mask = true_shift == 0

            # First alarm after the initial drift onset.
            post_onset_alarm_indices = np.where(
                alarms & (np.arange(len(alarms)) >= 20)
            )[0]

            first_alarm = (
                int(post_onset_alarm_indices[0])
                if len(post_onset_alarm_indices) else None
            )

            evaluation_rows.append({
                "condition": condition,
                "seed": seed,
                "detector": detector,
                "detection_rate": float(alarms[shifted_mask].mean()),
                "stable_false_alarm_rate": (
                    float(alarms[stable_mask].mean())
                    if stable_mask.any() else np.nan
                ),
                "first_alarm_window": first_alarm,
                "max_score": float(scores.max()),
                "n_alarms": int(alarms.sum())
            })

            for w in range(len(scores)):
                trace_rows.append({
                    "condition": condition,
                    "seed": seed,
                    "detector": detector,
                    "window": w,
                    "score": float(scores[w]),
                    "threshold": FROZEN_THRESHOLDS[detector],
                    "alarm": bool(alarms[w]),
                    "true_shift_intensity": float(true_shift[w])
                })

evaluation_df = pd.DataFrame(evaluation_rows)
trace_df = pd.DataFrame(trace_rows)

summary_df = (
    evaluation_df.groupby(["condition", "detector"])
    .agg(
        mean_detection_rate=("detection_rate", "mean"),
        sd_detection_rate=("detection_rate", "std"),
        mean_stable_false_alarm_rate=(
            "stable_false_alarm_rate", "mean"
        ),
        mean_alarms=("n_alarms", "mean"),
        mean_max_score=("max_score", "mean")
    )
    .reset_index()
)

display(summary_df)

evaluation_df.to_csv(
    OUTPUT_DIR / "temporal_drift_evaluation.csv", index=False
)
trace_df.to_csv(
    OUTPUT_DIR / "temporal_drift_trace.csv", index=False
)
summary_df.to_csv(
    OUTPUT_DIR / "temporal_drift_summary.csv", index=False
)

print("Temporal drift evaluation complete.")
print("Evaluation seeds:", len(EVALUATION_SEEDS))
print("Conditions:", evaluation_df["condition"].nunique())
print("Detectors:", evaluation_df["detector"].nunique())
print("Thresholds unchanged:", FROZEN_THRESHOLDS)
print("Saved evaluation, trace, and summary CSV files.")


,condition,detector,mean_detection_rate,sd_detection_rate,mean_stable_false_alarm_rate,mean_alarms,mean_max_score
0,abrupt,euclidean,1.000,0.000000,0.0675,41.35,4.850840
1,abrupt,mahalanobis,1.000,0.000000,0.0675,41.35,4.861030
2,gradual,euclidean,0.955,0.017396,0.0675,39.55,4.845867
3,gradual,mahalanobis,0.955,0.017396,0.0675,39.55,4.856046
4,recurring,euclidean,1.000,0.000000,0.0575,22.30,4.784236
5,recurring,mahalanobis,1.000,0.000000,0.0575,22.30,4.794286


Temporal drift evaluation complete.
Evaluation seeds: 20
Conditions: 3
Detectors: 2
Thresholds unchanged: {'euclidean': 2.0702929416972333, 'mahalanobis': 2.0746417163536846}
Saved evaluation, trace, and summary CSV files.


In [14]:

# Experiment 14: Cross-Detector Alarm Agreement

alarm_pivot = trace_df.pivot_table(
    index=["condition", "seed", "window"],
    columns="detector",
    values="alarm",
    aggfunc="first"
).reset_index()

alarm_pivot["same_alarm"] = (
    alarm_pivot["euclidean"] == alarm_pivot["mahalanobis"]
)

agreement_summary = (
    alarm_pivot.groupby("condition")
    .agg(
        total_window_comparisons=("same_alarm", "size"),
        matching_alarm_decisions=("same_alarm", "sum"),
        agreement_rate=("same_alarm", "mean")
    )
    .reset_index()
)

display(agreement_summary)

print(
    "Overall agreement:",
    f"{alarm_pivot['same_alarm'].mean():.4%}"
)

agreement_summary.to_csv(
    OUTPUT_DIR / "detector_alarm_agreement.csv",
    index=False
)

print("Saved detector_alarm_agreement.csv")


,condition,total_window_comparisons,matching_alarm_decisions,agreement_rate
0,abrupt,1200,1200,1.0
1,gradual,1200,1200,1.0
2,recurring,1200,1200,1.0


Overall agreement: 100.0000%
Saved detector_alarm_agreement.csv


In [15]:

# Experiment 14: Detection Delay Analysis
# Uses existing evaluation traces; no new streams or threshold changes.

delay_rows = []

for condition in ["abrupt", "gradual", "recurring"]:
    condition_trace = trace_df[trace_df["condition"] == condition]

    for seed in sorted(condition_trace["seed"].unique()):
        seed_trace = condition_trace[
            (condition_trace["seed"] == seed) &
            (condition_trace["detector"] == "euclidean")
        ].sort_values("window")

        alarms = seed_trace["alarm"].to_numpy(dtype=bool)

        if condition in ["abrupt", "gradual"]:
            onset = 20
            candidates = np.where(
                alarms & (np.arange(len(alarms)) >= onset)
            )[0]

            first_alarm = int(candidates[0]) if len(candidates) else None

            delay_rows.append({
                "condition": condition,
                "seed": seed,
                "episode": 1,
                "onset_window": onset,
                "first_alarm_window": first_alarm,
                "detection_delay_windows": (
                    first_alarm - onset if first_alarm is not None else np.nan
                ),
                "detected": first_alarm is not None
            })

        else:
            # Recurring episodes start at windows 20 and 40.
            # Only count alarms within each shifted episode.
            for episode, (start, end) in enumerate([(20, 30), (40, 50)], start=1):
                candidates = np.where(
                    alarms &
                    (np.arange(len(alarms)) >= start) &
                    (np.arange(len(alarms)) < end)
                )[0]

                first_alarm = int(candidates[0]) if len(candidates) else None

                delay_rows.append({
                    "condition": condition,
                    "seed": seed,
                    "episode": episode,
                    "onset_window": start,
                    "first_alarm_window": first_alarm,
                    "detection_delay_windows": (
                        first_alarm - start if first_alarm is not None else np.nan
                    ),
                    "detected": first_alarm is not None
                })

delay_df = pd.DataFrame(delay_rows)

delay_summary = (
    delay_df.groupby("condition")
    .agg(
        episodes=("detected", "size"),
        detected_episodes=("detected", "sum"),
        episode_detection_rate=("detected", "mean"),
        mean_delay_windows=("detection_delay_windows", "mean"),
        median_delay_windows=("detection_delay_windows", "median")
    )
    .reset_index()
)

display(delay_summary)

delay_df.to_csv(
    OUTPUT_DIR / "detection_delay_by_episode.csv", index=False
)
delay_summary.to_csv(
    OUTPUT_DIR / "detection_delay_summary.csv", index=False
)

print("Detection-delay analysis complete.")
print("Saved episode-level delays and summary.")
print("Note: an alarm before a recurring episode is not counted as detecting that episode.")


,condition,episodes,detected_episodes,episode_detection_rate,mean_delay_windows,median_delay_windows
0,abrupt,20,20,1.0,0.00,0.0
1,gradual,20,20,1.0,1.45,1.5
2,recurring,40,40,1.0,0.00,0.0


Detection-delay analysis complete.
Saved episode-level delays and summary.
Note: an alarm before a recurring episode is not counted as detecting that episode.


In [16]:

# Experiment 14: Recurring Drift Alarm Persistence Check

persistence_rows = []

for seed in sorted(
    trace_df.loc[trace_df["condition"] == "recurring", "seed"].unique()
):
    seed_trace = trace_df[
        (trace_df["condition"] == "recurring") &
        (trace_df["seed"] == seed) &
        (trace_df["detector"] == "euclidean")
    ].sort_values("window")

    alarms = seed_trace["alarm"].to_numpy(dtype=bool)

    persistence_rows.append({
        "seed": seed,
        "alarms_episode_1": int(alarms[20:30].sum()),
        "alarms_baseline_gap": int(alarms[30:40].sum()),
        "alarms_episode_2": int(alarms[40:50].sum()),
        "alarms_final_baseline": int(alarms[50:60].sum()),
        "gap_alarm_free": not alarms[30:40].any(),
        "final_baseline_alarm_free": not alarms[50:60].any()
    })

persistence_df = pd.DataFrame(persistence_rows)

display(persistence_df.head(10))

persistence_summary = persistence_df[
    [
        "alarms_episode_1",
        "alarms_baseline_gap",
        "alarms_episode_2",
        "alarms_final_baseline"
    ]
].mean().to_frame("mean_alarms_per_seed")

display(persistence_summary)

print(
    "Seeds with an alarm-free gap:",
    int(persistence_df["gap_alarm_free"].sum()),
    "out of", len(persistence_df)
)
print(
    "Seeds with an alarm-free final baseline:",
    int(persistence_df["final_baseline_alarm_free"].sum()),
    "out of", len(persistence_df)
)

persistence_df.to_csv(
    OUTPUT_DIR / "recurring_alarm_persistence.csv", index=False
)

print("Saved recurring_alarm_persistence.csv")


,seed,alarms_episode_1,alarms_baseline_gap,alarms_episode_2,alarms_final_baseline,gap_alarm_free,final_baseline_alarm_free
0,20261700,10,1,10,0,False,True
1,20261701,10,1,10,0,False,True
2,20261702,10,1,10,0,False,True
3,20261703,10,1,10,0,False,True
4,20261704,10,0,10,0,True,True
5,20261705,10,1,10,0,False,True
6,20261706,10,1,10,0,False,True
7,20261707,10,0,10,1,True,False
8,20261708,10,0,10,0,True,True
9,20261709,10,0,10,1,True,False


,mean_alarms_per_seed
alarms_episode_1,10.00
alarms_baseline_gap,0.60
alarms_episode_2,10.00
alarms_final_baseline,0.35


Seeds with an alarm-free gap: 9 out of 20
Seeds with an alarm-free final baseline: 13 out of 20
Saved recurring_alarm_persistence.csv


In [17]:

# Experiment 14: Recurring Episode Separation Summary

n_seeds = len(persistence_df)

separation_summary = pd.DataFrame([{
    "n_seeds": n_seeds,
    "both_shifted_episodes_all_windows_alarm": int(
        (
            (persistence_df["alarms_episode_1"] == 10) &
            (persistence_df["alarms_episode_2"] == 10)
        ).sum()
    ),
    "gap_completely_alarm_free": int(
        persistence_df["gap_alarm_free"].sum()
    ),
    "final_baseline_completely_alarm_free": int(
        persistence_df["final_baseline_alarm_free"].sum()
    ),
    "both_baseline_periods_alarm_free": int(
        (
            persistence_df["gap_alarm_free"] &
            persistence_df["final_baseline_alarm_free"]
        ).sum()
    )
}])

display(separation_summary)

separation_summary.to_csv(
    OUTPUT_DIR / "recurring_episode_separation_summary.csv",
    index=False
)

print("Saved recurring_episode_separation_summary.csv")


,n_seeds,both_shifted_episodes_all_windows_alarm,gap_completely_alarm_free,final_baseline_completely_alarm_free,both_baseline_periods_alarm_free
0,20,20,9,13,4


Saved recurring_episode_separation_summary.csv


## Step 21: Statistical Analysis Plan

### Experimental Unit
The independent random seed is the primary unit for comparisons across synthetic streams. Individual windows within a stream will not be treated as independent experimental replicates.

### Primary Outcomes
1. Detection rate across shifted windows.
2. Detection delay from the defined drift onset.
3. False-alarm rate during stable windows.
4. Successful separation of recurring drift episodes from stable intervals.

### Uncertainty
Report per-seed summaries and uncertainty estimates across seeds. Clearly distinguish variation across seeds from uncertainty due to finite windows within a stream.

### Detector Comparisons
Because Euclidean and Mahalanobis alarm decisions matched in the current evaluation, claims of detector superiority are not justified by these results. Any further comparison must examine whether their scores or outcomes differ under an appropriately designed experiment.

### Interpretation
Results describe the specified synthetic distributions and frozen thresholds only. They do not establish real-world biometric accuracy, fairness, or deployment readiness.

In [18]:

# Experiment 14: Consolidated Results Summary
# Uses existing outputs only.

print("=" * 65)
print("EXPERIMENT 14: FINAL RESULTS SUMMARY")
print("=" * 65)

print("\n1. FROZEN DETECTOR THRESHOLDS")
for detector, threshold in FROZEN_THRESHOLDS.items():
    print(f"{detector}: {threshold:.6f}")

print("\n2. INDEPENDENT BASELINE CONTROL")
display(control_summary)

print("\n3. TEMPORAL DRIFT EVALUATION")
display(summary_df)

print("\n4. DETECTOR ALARM AGREEMENT")
display(agreement_summary)

print("\n5. DETECTION DELAY")
display(delay_summary)

print("\n6. RECURRING-DRIFT SEPARATION")
display(separation_summary)

print("\n7. OUTPUT DIRECTORY")
print(OUTPUT_DIR.resolve())

print("\nSummary generated from existing experiment outputs.")
print("No new data generated and no thresholds modified.")


EXPERIMENT 14: FINAL RESULTS SUMMARY

1. FROZEN DETECTOR THRESHOLDS
euclidean: 2.070293
mahalanobis: 2.074642

2. INDEPENDENT BASELINE CONTROL


,detector,mean_false_alarm_rate,sd_across_seeds,total_alarms,total_windows
0,euclidean,0.0497,0.007794,497,10000
1,mahalanobis,0.0497,0.007794,497,10000



3. TEMPORAL DRIFT EVALUATION


,condition,detector,mean_detection_rate,sd_detection_rate,mean_stable_false_alarm_rate,mean_alarms,mean_max_score
0,abrupt,euclidean,1.000,0.000000,0.0675,41.35,4.850840
1,abrupt,mahalanobis,1.000,0.000000,0.0675,41.35,4.861030
2,gradual,euclidean,0.955,0.017396,0.0675,39.55,4.845867
3,gradual,mahalanobis,0.955,0.017396,0.0675,39.55,4.856046
4,recurring,euclidean,1.000,0.000000,0.0575,22.30,4.784236
5,recurring,mahalanobis,1.000,0.000000,0.0575,22.30,4.794286



4. DETECTOR ALARM AGREEMENT


...


5. DETECTION DELAY


,condition,episodes,detected_episodes,episode_detection_rate,mean_delay_windows,median_delay_windows
0,abrupt,20,20,1.0,0.00,0.0
1,gradual,20,20,1.0,1.45,1.5
2,recurring,40,40,1.0,0.00,0.0



6. RECURRING-DRIFT SEPARATION


,n_seeds,both_shifted_episodes_all_windows_alarm,gap_completely_alarm_free,final_baseline_completely_alarm_free,both_baseline_periods_alarm_free
0,20,20,9,13,4



7. OUTPUT DIRECTORY
/content/results/experiment_14

Summary generated from existing experiment outputs.
No new data generated and no thresholds modified.


In [19]:

# Experiment 14: Artifact Inventory and Integrity Check

from pathlib import Path
import json
import pandas as pd

output_dir = Path("/content/results/experiment_14")

expected_files = [
    "environment.json",
    "calibration_scores.csv",
    "frozen_thresholds.json",
    "independent_control_results.csv",
    "independent_control_summary.csv",
    "baseline_detector_score_comparison.csv",
    "temporal_drift_evaluation.csv",
    "temporal_drift_trace.csv",
    "temporal_drift_summary.csv",
    "detector_alarm_agreement.csv",
    "detection_delay_by_episode.csv",
    "detection_delay_summary.csv",
    "recurring_alarm_persistence.csv",
    "recurring_episode_separation_summary.csv",
]

inventory = []

for filename in expected_files:
    path = output_dir / filename
    inventory.append({
        "file": filename,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

print("Files present:", int(inventory_df["exists"].sum()), "of", len(expected_files))
print("Output directory:", output_dir.resolve())

# Validate the frozen threshold record if present.
threshold_path = output_dir / "frozen_thresholds.json"

if threshold_path.exists():
    with open(threshold_path) as f:
        saved_thresholds = json.load(f)

    print("\nSaved threshold record:")
    print(json.dumps(saved_thresholds, indent=2))

# Validate key CSV row counts if present.
for filename in [
    "temporal_drift_evaluation.csv",
    "temporal_drift_trace.csv",
    "independent_control_results.csv"
]:
    path = output_dir / filename
    if path.exists():
        df = pd.read_csv(path)
        print(f"{filename}: {len(df)} rows, {len(df.columns)} columns")

inventory_df.to_csv(output_dir / "artifact_inventory.csv", index=False)
print("\nSaved artifact_inventory.csv")


,file,exists,size_bytes
0,environment.json,True,267
1,calibration_scores.csv,True,8215
2,frozen_thresholds.json,True,272
3,independent_control_results.csv,True,1362
4,independent_control_summary.csv,True,172
5,baseline_detector_score_comparison.csv,True,26669
6,temporal_drift_evaluation.csv,True,7472
7,temporal_drift_trace.csv,True,561319
8,temporal_drift_summary.csv,True,550
9,detector_alarm_agreement.csv,True,142


Files present: 14 of 14
Output directory: /content/results/experiment_14

Saved threshold record:
{
  "target_false_alarm_rate": 0.05,
  "calibration_windows": 200,
  "thresholds": {
    "euclidean": 2.0702929416972333,
    "mahalanobis": 2.0746417163536846
  },
  "method": "Empirical upper-tail quantile",
  "reference_seed": 20261401,
  "calibration_seed": 20261402
}
temporal_drift_evaluation.csv: 120 rows, 8 columns
temporal_drift_trace.csv: 7200 rows, 8 columns
independent_control_results.csv: 40 rows, 5 columns

Saved artifact_inventory.csv


In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

output_dir = Path("/content/results/experiment_14")

# 1. Confirm key columns and inspect the stored summaries.
# Fixed: Removed 'alarm' from temporal_drift_evaluation.csv expected columns
files_to_check = {
    "temporal_drift_evaluation.csv": [
        "condition", "detector", "seed",
        "detection_rate", "stable_false_alarm_rate"
    ],
    "temporal_drift_trace.csv": [
        "condition", "detector", "seed", "window",
        "true_shift_intensity", "alarm", "score"
    ],
    "independent_control_results.csv": [
        "detector", "seed"
    ],
}

for filename, required_columns in files_to_check.items():
    df = pd.read_csv(output_dir / filename)
    missing = sorted(set(required_columns) - set(df.columns))
    print(f"\n{filename}")
    print("Rows:", len(df))
    print("Missing required columns:", missing)
    print("Duplicate rows:", int(df.duplicated().sum()))

# 2. Verify alarm agreement directly from the temporal trace.
trace = pd.read_csv(output_dir / "temporal_drift_trace.csv")

alarm_pivot = trace.pivot_table(
    index=["condition", "seed", "window"],
    columns="detector",
    values="alarm",
    aggfunc="first"
)

if {"euclidean", "mahalanobis"}.issubset(alarm_pivot.columns):
    agreement = (
        alarm_pivot["euclidean"].astype(bool)
        == alarm_pivot["mahalanobis"].astype(bool)
    )
    print("\nTemporal alarm agreement:")
    print(f"Matching decisions: {int(agreement.sum())}/{len(agreement)}")
    print(f"Agreement rate: {agreement.mean():.4%}")
else:
    print("\nCould not compare alarms: expected detector names not found.")

# 3. Recalculate stable-period false-alarm rates from raw trace.
stable = trace[trace["true_shift_intensity"] == 0].copy()
stable_rates = (
    stable.groupby(["condition", "detector", "seed"])["alarm"]
    .mean()
    .reset_index(name="recalculated_stable_false_alarm_rate")
)

print("\nRecalculated stable-period false-alarm rates:")
display(
    stable_rates.groupby(["condition", "detector"])
    ["recalculated_stable_false_alarm_rate"]
    .agg(["mean", "std", "count"]).round(4)
)

# 4. Check whether the saved summary files are present.
for filename in [
    "temporal_drift_summary.csv",
    "detection_delay_summary.csv",
    "recurring_episode_separation_summary.csv",
    "independent_control_summary.csv",
]:
    path = output_dir / filename
    print(f"{filename}: {'present' if path.is_file() else 'MISSING'}")

print("\nVerification finished. No experiment data or thresholds were changed.")


temporal_drift_evaluation.csv
Rows: 120
Missing required columns: []
Duplicate rows: 0

temporal_drift_trace.csv
Rows: 7200
Missing required columns: []
Duplicate rows: 0

independent_control_results.csv
Rows: 40
Missing required columns: []
Duplicate rows: 0

Temporal alarm agreement:
Matching decisions: 3600/3600
Agreement rate: 100.0000%

Recalculated stable-period false-alarm rates:


mean     std  count
condition detector                          
abrupt    euclidean    0.0675  0.0568     20
          mahalanobis  0.0675  0.0568     20
gradual   euclidean    0.0675  0.0568     20
          mahalanobis  0.0675  0.0568     20
recurring euclidean    0.0575  0.0335     20
          mahalanobis  0.0575  0.0335     20

temporal_drift_summary.csv: present
detection_delay_summary.csv: present
recurring_episode_separation_summary.csv: present
independent_control_summary.csv: present

Verification finished. No experiment data or thresholds were changed.


In [24]:
from pathlib import Path
import pandas as pd

output_dir = Path("/content/results/experiment_14")
trace = pd.read_csv(output_dir / "temporal_drift_trace.csv")

# Stable windows have no imposed shift.
# Fixed: Changed column reference from 'true_shift' to 'true_shift_intensity'
stable = trace[trace["true_shift_intensity"] == 0].copy()

per_seed = (
    stable.groupby(["condition", "detector", "seed"]
)
    .agg(
        stable_windows=("alarm", "size"),
        false_alarms=("alarm", "sum"),
        false_alarm_rate=("alarm", "mean")
    )
    .reset_index()
)

print("Stable-window false alarms by seed:")
display(per_seed.round(4))

print("\nDistribution across seeds:")
summary = (
    per_seed.groupby(["condition", "detector"])["false_alarm_rate"]
    .agg(
        mean="mean",
        sd="std",
        minimum="min",
        median="median",
        maximum="max"
    )
    .reset_index()
)
display(summary.round(4))

# Count seeds exceeding the nominal 5% target.
per_seed["exceeds_5_percent"] = per_seed["false_alarm_rate"] > 0.05

print("\nSeeds with stable-period FPR above 5%:")
display(
    per_seed.groupby(["condition", "detector"])["exceeds_5_percent"]
    .sum()
    .rename("seeds_above_target")
    .reset_index()
)

per_seed.to_csv(
    output_dir / "stable_false_alarm_by_seed.csv",
    index=False
)
summary.to_csv(
    output_dir / "stable_false_alarm_seed_summary.csv",
    index=False
)

print("\nSaved stable_false_alarm_by_seed.csv")
print("Saved stable_false_alarm_seed_summary.csv")
print("No thresholds or experiment streams were modified.")

Stable-window false alarms by seed:


,condition,detector,seed,stable_windows,false_alarms,false_alarm_rate
0,abrupt,euclidean,20261700,20,3,0.150
1,abrupt,euclidean,20261701,20,2,0.100
2,abrupt,euclidean,20261702,20,3,0.150
3,abrupt,euclidean,20261703,20,1,0.050
4,abrupt,euclidean,20261704,20,0,0.000
...,...,...,...,...,...,...
115,recurring,mahalanobis,20261715,40,2,0.050
116,recurring,mahalanobis,20261716,40,3,0.075
117,recurring,mahalanobis,20261717,40,0,0.000
118,recurring,mahalanobis,20261718,40,3,0.075



Distribution across seeds:


,condition,detector,mean,sd,minimum,median,maximum
0,abrupt,euclidean,0.0675,0.0568,0.0,0.05,0.15
1,abrupt,mahalanobis,0.0675,0.0568,0.0,0.05,0.15
2,gradual,euclidean,0.0675,0.0568,0.0,0.05,0.15
3,gradual,mahalanobis,0.0675,0.0568,0.0,0.05,0.15
4,recurring,euclidean,0.0575,0.0335,0.0,0.05,0.10
5,recurring,mahalanobis,0.0575,0.0335,0.0,0.05,0.10



Seeds with stable-period FPR above 5%:


,condition,detector,seeds_above_target
0,abrupt,euclidean,9
1,abrupt,mahalanobis,9
2,gradual,euclidean,9
3,gradual,mahalanobis,9
4,recurring,euclidean,9
5,recurring,mahalanobis,9



Saved stable_false_alarm_by_seed.csv
Saved stable_false_alarm_seed_summary.csv
No thresholds or experiment streams were modified.


In [25]:

from pathlib import Path
import pandas as pd

output_dir = Path("/content/results/experiment_14")
trace = pd.read_csv(output_dir / "temporal_drift_trace.csv")

# Use one detector because Euclidean and Mahalanobis alarm decisions
# were previously verified to match exactly.
euclidean = trace[trace["detector"] == "euclidean"].copy()

segments = {
    "shifted_episode_1": (20, 29),
    "stable_gap": (30, 39),
    "shifted_episode_2": (40, 49),
    "final_stable_period": (50, 59),
}

rows = []

for (condition, seed), group in euclidean.groupby(["condition", "seed"]):
    for segment, (start, end) in segments.items():
        part = group[group["window"].between(start, end)]
        if part.empty:
            continue

        rows.append({
            "condition": condition,
            "seed": seed,
            "segment": segment,
            "n_windows": len(part),
            "alarm_count": int(part["alarm"].sum()),
            "alarm_rate": float(part["alarm"].mean()),
            "alarm_free": bool(part["alarm"].sum() == 0),
        })

segment_df = pd.DataFrame(rows)

print("Alarm behaviour by segment:")
display(
    segment_df.groupby(["condition", "segment"]).agg(
        seeds=("seed", "nunique"),
        mean_alarm_rate=("alarm_rate", "mean"),
        mean_alarm_count=("alarm_count", "mean"),
        alarm_free_seeds=("alarm_free", "sum")
    ).reset_index().round(4)
)

segment_df.to_csv(
    output_dir / "recurring_segment_alarm_diagnostic.csv",
    index=False
)

print("\nSaved recurring_segment_alarm_diagnostic.csv")
print("No thresholds or experiment streams were modified.")


Alarm behaviour by segment:


,condition,segment,seeds,mean_alarm_rate,mean_alarm_count,alarm_free_seeds
0,abrupt,final_stable_period,20,1.000,10.00,0
1,abrupt,shifted_episode_1,20,1.000,10.00,0
2,abrupt,shifted_episode_2,20,1.000,10.00,0
3,abrupt,stable_gap,20,1.000,10.00,0
4,gradual,final_stable_period,20,1.000,10.00,0
5,gradual,shifted_episode_1,20,0.820,8.20,0
6,gradual,shifted_episode_2,20,1.000,10.00,0
7,gradual,stable_gap,20,1.000,10.00,0
8,recurring,final_stable_period,20,0.035,0.35,13
9,recurring,shifted_episode_1,20,1.000,10.00,0



Saved recurring_segment_alarm_diagnostic.csv
No thresholds or experiment streams were modified.


In [27]:

from pathlib import Path
import pandas as pd

output_dir = Path("/content/results/experiment_14")
trace = pd.read_csv(output_dir / "temporal_drift_trace.csv")

print("Column names:")
print(trace.columns.tolist())

print("\nData types:")
print(trace.dtypes)

print("\nAlarm values as stored:")
print(trace["alarm"].value_counts(dropna=False).to_string())

print("\nAlarm values by condition and detector:")
display(pd.crosstab(
    [trace["condition"], trace["detector"]],
    trace["alarm"],
    dropna=False
))

print("\nRaw Euclidean rows for abrupt, seed 20261700:")
display(
    trace[
        (trace["condition"] == "abrupt") &
        (trace["detector"] == "euclidean") &
        (trace["seed"] == 20261700)
    ][["window", "true_shift_intensity", "score", "alarm"]]
    .sort_values("window")
    .to_string(index=False)
)

print("\nNo data or thresholds modified.")


Column names:
['condition', 'seed', 'detector', 'window', 'score', 'threshold', 'alarm', 'true_shift_intensity']

Data types:
condition                object
seed                      int64
detector                 object
window                    int64
score                   float64
threshold               float64
alarm                      bool
true_shift_intensity    float64
dtype: object

Alarm values as stored:
alarm
True     4128
False    3072

Alarm values by condition and detector:


alarm                  False  True 
condition detector                 
abrupt    euclidean      373    827
          mahalanobis    373    827
gradual   euclidean      409    791
          mahalanobis    409    791
recurring euclidean      754    446
          mahalanobis    754    446


Raw Euclidean rows for abrupt, seed 20261700:


' window  true_shift_intensity    score  alarm\n      0                   0.0 1.951492  False\n      1                   0.0 1.836322  False\n      2                   0.0 2.176175   True\n      3                   0.0 1.950098  False\n      4                   0.0 1.773177  False\n      5                   0.0 1.925650  False\n      6                   0.0 1.778159  False\n      7                   0.0 1.846544  False\n      8                   0.0 2.254189   True\n      9                   0.0 1.910612  False\n     10                   0.0 1.553799  False\n     11                   0.0 1.732641  False\n     12                   0.0 1.953472  False\n     13                   0.0 2.077703   True\n     14                   0.0 1.645065  False\n     15                   0.0 1.797314  False\n     16                   0.0 1.905072  False\n     17                   0.0 1.791174  False\n     18                   0.0 2.042782  False\n     19                   0.0 1.789125  False\n     20     


No data or thresholds modified.


In [28]:

from pathlib import Path
import pandas as pd

output_dir = Path("/content/results/experiment_14")
trace = pd.read_csv(output_dir / "temporal_drift_trace.csv")

# Use Euclidean only because detector alarm decisions matched exactly.
df = trace[trace["detector"] == "euclidean"].copy()

# Use actual shift intensity, not fixed window ranges, to identify
# shifted and stable windows.
df["segment"] = "stable"
df.loc[df["true_shift_intensity"] > 0, "segment"] = "shifted"

# Retain episode labels for recurring drift.
df.loc[
    df["condition"].eq("recurring") & df["window"].between(20, 29),
    "segment"
] = "shifted_episode_1"

df.loc[
    df["condition"].eq("recurring") & df["window"].between(30, 39),
    "segment"
] = "stable_gap"

df.loc[
    df["condition"].eq("recurring") & df["window"].between(40, 49),
    "segment"
] = "shifted_episode_2"

df.loc[
    df["condition"].eq("recurring") & df["window"].between(50, 59),
    "segment"
] = "final_stable_period"

# Summarize observed alarms for each condition and segment.
summary = (
    df.groupby(["condition", "segment"])
    .agg(
        n_windows=("alarm", "size"),
        mean_alarm_rate=("alarm", "mean"),
        mean_alarm_count_per_seed=("alarm", "sum"),
        alarm_free_windows=("alarm", lambda s: int((~s).sum()))
    )
    .reset_index()
)

# Also calculate the number of seeds with no alarms in each segment.
seed_summary = (
    df.groupby(["condition", "seed", "segment"])["alarm"]
    .agg(alarm_count="sum", n_windows="size")
    .reset_index()
)
seed_summary["alarm_free"] = seed_summary["alarm_count"] == 0

free_seeds = (
    seed_summary.groupby(["condition", "segment"])["alarm_free"]
    .sum()
    .reset_index(name="alarm_free_seeds")
)

summary = summary.merge(
    free_seeds, on=["condition", "segment"], how="left"
)

display(summary.round(4))

summary.to_csv(
    output_dir / "corrected_segment_alarm_diagnostic.csv",
    index=False
)

print("\nSaved corrected_segment_alarm_diagnostic.csv")
print("No streams or thresholds were changed.")


,condition,segment,n_windows,mean_alarm_rate,mean_alarm_count_per_seed,alarm_free_windows,alarm_free_seeds
0,abrupt,shifted,800,1.0000,800,0,0
1,abrupt,stable,400,0.0675,27,373,6
2,gradual,shifted,800,0.9550,764,36,0
3,gradual,stable,400,0.0675,27,373,6
4,recurring,final_stable_period,200,0.0350,7,193,13
5,recurring,shifted_episode_1,200,1.0000,200,0,0
6,recurring,shifted_episode_2,200,1.0000,200,0,0
7,recurring,stable,400,0.0675,27,373,6
8,recurring,stable_gap,200,0.0600,12,188,9



Saved corrected_segment_alarm_diagnostic.csv
No streams or thresholds were changed.


In [29]:

from pathlib import Path
import pandas as pd
import json
import hashlib

output_dir = Path("/content/results/experiment_14")

# Verify the key artifacts from Experiment 14.
expected_files = [
    "environment.json",
    "frozen_thresholds.json",
    "independent_control_results.csv",
    "independent_control_summary.csv",
    "baseline_detector_score_comparison.csv",
    "temporal_drift_evaluation.csv",
    "temporal_drift_trace.csv",
    "temporal_drift_summary.csv",
    "detector_alarm_agreement.csv",
    "detection_delay_by_episode.csv",
    "detection_delay_summary.csv",
    "recurring_alarm_persistence.csv",
    "recurring_episode_separation_summary.csv",
    "stable_false_alarm_by_seed.csv",
    "stable_false_alarm_seed_summary.csv",
    "recurring_segment_alarm_diagnostic.csv",
    "corrected_segment_alarm_diagnostic.csv",
]

inventory = []
for filename in expected_files:
    path = output_dir / filename
    inventory.append({
        "file": filename,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
        "sha256": (
            hashlib.sha256(path.read_bytes()).hexdigest()
            if path.is_file() else None
        )
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df[["file", "exists", "size_bytes"]])

print(
    "\nArtifacts present:",
    int(inventory_df["exists"].sum()),
    "of",
    len(expected_files)
)

# Confirm frozen thresholds remain unchanged.
with open(output_dir / "frozen_thresholds.json") as f:
    thresholds = json.load(f)

print("\nFrozen thresholds:")
print(json.dumps(thresholds, indent=2))

# Save an integrity manifest with file hashes.
manifest_path = output_dir / "experiment_14_integrity_manifest.csv"
inventory_df.to_csv(manifest_path, index=False)

print("\nSaved integrity manifest:", manifest_path)
print("No experiment data, thresholds, or evaluation results were changed.")


,file,exists,size_bytes
0,environment.json,True,267
1,frozen_thresholds.json,True,272
2,independent_control_results.csv,True,1362
3,independent_control_summary.csv,True,172
4,baseline_detector_score_comparison.csv,True,26669
5,temporal_drift_evaluation.csv,True,7472
6,temporal_drift_trace.csv,True,561319
7,temporal_drift_summary.csv,True,550
8,detector_alarm_agreement.csv,True,142
9,detection_delay_by_episode.csv,True,2708



Artifacts present: 17 of 17

Frozen thresholds:
{
  "target_false_alarm_rate": 0.05,
  "calibration_windows": 200,
  "thresholds": {
    "euclidean": 2.0702929416972333,
    "mahalanobis": 2.0746417163536846
  },
  "method": "Empirical upper-tail quantile",
  "reference_seed": 20261401,
  "calibration_seed": 20261402
}

Saved integrity manifest: /content/results/experiment_14/experiment_14_integrity_manifest.csv
No experiment data, thresholds, or evaluation results were changed.


# Experiment 14: Drift Detection Under Gradual and Recurring Distribution Shifts

## Objective

Evaluate Euclidean and regularized Mahalanobis embedding-level drift detectors under controlled abrupt, gradual, and recurring distribution shifts, focusing on detection behaviour, stable-period false alarms, detection delay, and alarm persistence.

## Experimental Setup

- Embedding dimension: 64
- Window size: 20 samples
- Stream length: 60 windows
- Independent evaluation seeds: 20 per drift condition
- Target calibration false-alarm rate: 5%
- Detectors: Euclidean distance and regularized Mahalanobis distance
- Thresholds: frozen after calibration using 200 calibration windows
- Reference and calibration seeds: 20261401 and 20261402
- Evaluation seed range: 20261700–20261719

## Main Results

### Detection during shifted periods

| Condition | Detector | Mean detection rate |
|---|---|---:|
| Abrupt | Euclidean | 100.0% |
| Abrupt | Mahalanobis | 100.0% |
| Gradual | Euclidean | 95.5% |
| Gradual | Mahalanobis | 95.5% |
| Recurring | Euclidean | 100.0% |
| Recurring | Mahalanobis | 100.0% |

### Stable-period false alarms

The independent control evaluation measured a mean false-alarm rate of 4.97% across 10,000 baseline windows per detector, with a standard deviation of 0.78 percentage points across seeds.

In the temporal drift evaluation, the observed stable-period false-alarm rates were 6.75% for abrupt and gradual conditions and 5.75% for recurring conditions. These are empirical estimates and should not be interpreted as proof that the nominal 5% rate is consistently achieved.

### Detector agreement

Euclidean and Mahalanobis detectors produced identical alarm decisions across all 3,600 condition-seed-window comparisons in the temporal evaluation. Their scores were strongly correlated in the baseline diagnostic.

Consequently, this experiment does not establish superiority of either detector. The similarity may be influenced by the controlled, approximately isotropic synthetic distribution and the estimated covariance structure.

### Recurring drift and alarm persistence

Both imposed recurring drift episodes achieved a 100% mean alarm rate. During the stable gap between episodes, the mean alarm rate was 6.0%; 9 of 20 seeds had an entirely alarm-free gap. During the final stable period, the mean alarm rate was 3.5%; 13 of 20 seeds had no alarms.

These results suggest that the detectors were sensitive to the imposed shifts but did not consistently separate recurring episodes with an alarm-free interval.

### Detection delay

The recorded episode-level analysis reported:
- Abrupt: mean delay of 0 windows.
- Gradual: mean delay of 1.45 windows.
- Recurring: mean delay of 0 windows per episode under the implemented episode-level definition.

These values depend on the experiment's operational definition of detection delay. In particular, recurring-episode detection should not be interpreted as proof that the detector reset or independently recognized a new event after clearing the previous alarm.

## Reproducibility and Artifacts

Results are stored under `results/experiment_14/`.

Key artifacts include:
- `environment.json`
- `frozen_thresholds.json`
- `independent_control_results.csv`
- `independent_control_summary.csv`
- `temporal_drift_evaluation.csv`
- `temporal_drift_trace.csv`
- `temporal_drift_summary.csv`
- `detector_alarm_agreement.csv`
- `detection_delay_by_episode.csv`
- `detection_delay_summary.csv`
- `stable_false_alarm_by_seed.csv`
- `stable_false_alarm_seed_summary.csv`
- `recurring_segment_alarm_diagnostic.csv`
- `corrected_segment_alarm_diagnostic.csv`
- `experiment_14_integrity_manifest.csv`

The thresholds and experiment streams were not modified during the final diagnostic checks. The corrected segment analysis uses the actual shift-intensity labels to distinguish stable and shifted windows.

## Limitations

1. The drift streams are synthetic and do not establish real-world biometric recognition accuracy, fairness, security, or deployment readiness.
2. Euclidean and Mahalanobis alarm decisions were identical in this experiment, limiting conclusions about comparative detector performance.
3. Stable-period false-alarm rates varied across seeds and conditions.
4. Episode-level delay and alarm persistence depend on the operational definition of an alarm and event.
5. Results are limited to the specified synthetic distributions, shift direction, shift magnitude, calibration procedure, and detector configurations.

## Conclusion

Experiment 14 provides a controlled evaluation of embedding-level drift detection under three temporal shift patterns. Both detectors were highly sensitive to the imposed shifts, with lower detection rates during gradual transitions. Stable-period false alarms and recurring-episode separation remain important limitations. The results are preliminary synthetic evidence, not validation of a real-world biometric monitoring system.